# pLM Covariance Pooling — Matrix-Power (**sqrt**) Experiments ONLY

**PP1 SoSe2026 | TU München**

Runs **only** the matrix-power-normalised (`C → C^{1/2}`, iSQRT-COV) pooling — the "sqrt" runs that are **missing** from the Drive results:

| Config | Method | `power_norm` |
|---|---|---|
| `cov_supervised_sqrt` | supervised covariance + `C → C^{1/2}` | **true** |
| `attention_cov_sqrt`  | light-attention covariance + `C → C^{1/2}` (full stack) | **true** |

…on both tasks (DeepLoc + Meltome), **`dc = 32` only, 1 seed (seed 0)** — minimal compute. Scale up later by editing `SEEDS` / adding a dc sweep in cell 7.

> ⚠️ **Why this notebook exists.** The earlier "full run" set `power_norm = False` for *every* config, which silently disables the matrix square root — so no valid sqrt result was ever produced. This notebook:
> 1. **asserts** `power_norm = true` in every config *before* running (cell 6),
> 2. writes **task-prefixed** output names so `scl` and `meltome` don't overwrite each other (they share the config stem `cov_supervised_sqrt` / `attention_cov_sqrt`),
> 3. **verifies** `power_norm` is ON in every output JSON *afterwards* (cell 9).

**Baselines for comparison:** `cov_supervised` (non-sqrt) is already on the Drive at dc=32, so `cov_supervised_sqrt` is directly comparable. `attention_cov` (non-sqrt) is **not** on the Drive — uncomment it in cell 7 if you want the ablation for `attention_cov_sqrt`.

### Workflow
1. GPU check · 2. Mount Drive · 3. Paths · 4. Install repo · 5. Wire Drive · **6. Pre-flight `power_norm` guard** · 7. Run sqrt configs (dc=32, seed 0) · 8. Mirror to Drive · **9. Verify `power_norm` ON + summary**

## 1 · GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    print('No GPU detected — training will be slow on CPU.\n'
          'Runtime -> Change runtime type -> GPU recommended.')
else:
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU:  {gpu.name}')
    print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')
    print(f'CUDA: {torch.version.cuda}')

## 2 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print('Top-level folders in MyDrive:')
for f in sorted(os.listdir('/content/drive/MyDrive')):
    print(f'  {f}')

## 3 · Configuration

Same two artefacts the core-grid notebook used. Adjust the paths if your Drive names differ.

In [ ]:
from pathlib import Path

# ───────────────────────────────────────────────────────────
EMB_DIR     = Path('/content/drive/MyDrive/embeddings')                       # shared shortcut
LABELS_ZIP  = Path('/content/drive/MyDrive/raw-20260516T183459Z-3-001.zip')   # uploaded once
CODE_ZIP    = Path('/content/drive/MyDrive/sop_repo.zip')                     # fallback if git clone fails
RESULTS_OUT = Path('/content/drive/MyDrive/pp1_sop_results')                  # where runs/ lands
# ───────────────────────────────────────────────────────────

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

checks = {
    'EMB_DIR (folder)':  (EMB_DIR, EMB_DIR.is_dir()),
    'LABELS_ZIP (file)': (LABELS_ZIP, LABELS_ZIP.is_file()),
}
print(f'{"Item":<22} {"Status":<10} Path')
print('-' * 90)
for name, (path, ok) in checks.items():
    print(f'{name:<22} {"OK" if ok else "MISSING":<10} {path}')

if EMB_DIR.is_dir():
    found = sorted(EMB_DIR.iterdir())
    print(f'\nEMB_DIR contains {len(found)} files: {[p.name for p in found]}')

## 4 · Install repo

Tries `git clone` first (works if the repo is public). Falls back to `sop_repo.zip` from Drive or a local upload picker.

To build the zip locally: `zip -r sop_repo.zip src scripts configs pyproject.toml README.md tests`

In [ ]:
import subprocess, sys, shutil, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/Julius-Schmidt/pLM-covariance-pooling.git'
REPO_DIR = Path('/content/pLM-covariance-pooling')
LOCAL_ZIP = Path('/content/sop_repo.zip')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# Try public git clone first — fast and always up to date.
result = subprocess.run(
    ['git', 'clone', '--depth=1', REPO_URL, str(REPO_DIR)],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print(f'Cloned from {REPO_URL}')
else:
    # Fall back to zip (Drive → local → picker).
    if CODE_ZIP.is_file():
        src_zip = CODE_ZIP
        print(f'Using zip from Drive: {CODE_ZIP}')
    elif LOCAL_ZIP.is_file():
        src_zip = LOCAL_ZIP
        print(f'Using local zip: {LOCAL_ZIP}')
    else:
        from google.colab import files
        print('Select sop_repo.zip ...')
        uploaded = files.upload()
        src_zip = Path(f'/content/{next(iter(uploaded))}')

    REPO_DIR.mkdir(parents=True)
    with zipfile.ZipFile(src_zip) as zf:
        zf.extractall(REPO_DIR)

    # Flatten if the zip added a top-level folder.
    if not (REPO_DIR / 'pyproject.toml').exists():
        inner = next(
            (p for p in REPO_DIR.iterdir() if p.is_dir() and (p / 'pyproject.toml').exists()),
            None,
        )
        if inner is None:
            raise RuntimeError('pyproject.toml not found in zip — check archive layout.')
        for item in inner.iterdir():
            shutil.move(str(item), REPO_DIR / item.name)
        inner.rmdir()
    print('Extracted from zip.')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
    cwd=str(REPO_DIR), check=True,
)
print(f'Package installed at {REPO_DIR}')

## 5 · Wire Drive into the repo

Configs use relative paths (`data/embeddings/*.h5`, `data/raw/*/`). This cell symlinks `data/embeddings/` → the shared Drive shortcut and extracts the labels zip into `data/raw/`. (No `models/` link needed — the sqrt methods load no pre-fitted checkpoints; they train end-to-end.)

In [ ]:
import shutil, zipfile
from pathlib import Path

DATA_DIR   = REPO_DIR / 'data'
EMB_LINK   = DATA_DIR / 'embeddings'
RAW_LINK   = DATA_DIR / 'raw'
LABELS_OUT = Path('/content/labels')

DATA_DIR.mkdir(exist_ok=True)

def relink(link: Path, target: Path):
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target)

# embeddings → Drive shortcut
relink(EMB_LINK, EMB_DIR)

# raw → extract labels zip locally, find the deeploc/meltome parent, symlink to it
if LABELS_OUT.exists():
    shutil.rmtree(LABELS_OUT)
LABELS_OUT.mkdir()
with zipfile.ZipFile(LABELS_ZIP) as zf:
    zf.extractall(LABELS_OUT)
raw_root = next(p.parent for p in LABELS_OUT.rglob('deeploc') if p.is_dir())
relink(RAW_LINK, raw_root)

# Verify the eight key paths the configs reference
for rel in ['data/embeddings/deeploc_train.h5',
            'data/embeddings/deeploc_test.h5',
            'data/embeddings/meltome_train.h5',
            'data/embeddings/meltome_test.h5',
            'data/raw/deeploc/train_labels.csv',
            'data/raw/deeploc/test_labels.csv',
            'data/raw/meltome/train_labels.csv',
            'data/raw/meltome/test_labels.csv']:
    full = REPO_DIR / rel
    print(f'{rel}: {"OK" if full.exists() else "MISSING"}')

## 6 · Pre-flight guard — `power_norm` must be `true`

This is the check that was missing last time. If any sqrt config does **not** have `power_norm: true`, the matrix square root `C → C^{1/2}` would be silently skipped and the run would just reproduce the plain covariance. The assert below stops the notebook before wasting GPU time on a nullified experiment.

In [ ]:
import yaml

# (task_dir, config_filename) — the matrix-power "sqrt" configs only.
SQRT_CONFIGS = [
    ('scl',     'cov_supervised_sqrt.yaml'),
    ('scl',     'attention_cov_sqrt.yaml'),
    ('meltome', 'cov_supervised_sqrt.yaml'),
    ('meltome', 'attention_cov_sqrt.yaml'),
]

print(f'{"config":<42}{"method":<18}{"power_norm"}')
print('-' * 74)
all_ok = True
for task, fname in SQRT_CONFIGS:
    cfg = yaml.safe_load((REPO_DIR / 'configs' / task / fname).read_text())
    pn = cfg['pooling'].get('power_norm', False)
    print(f'{task + "/" + fname:<42}{cfg["pooling"]["method"]:<18}{pn}')
    all_ok &= bool(pn)

assert all_ok, (
    'A sqrt config has power_norm != true — the matrix square root would be '
    'DISABLED (this is exactly the bug from the previous run). Fix the config.'
)
print('\n✓ All sqrt configs have power_norm = true  →  C → C^{1/2} will be applied.')

## 7 · Run the sqrt experiments  ·  dc = 32, seed 0 only

Minimal-compute setting: **`dc = 32`** (each config's default — no sweep) and **one seed**. `run_experiment.py` has no `--seeds` flag, so the seed count is set by overriding `seeds` in the task-prefixed config copy this cell writes.

The task prefix (`scl_… / meltome_…`) is needed because `scl` and `meltome` share the same config stem and would otherwise overwrite each other's JSON (output name is `{config_stem}_{method}_dc{dc}.json`). It also matches the existing `scl_* / meltome_*` naming on the Drive.

To scale up later: set `SEEDS = [0, 1, 2]`, and/or add `--dc 8 16 24 32 48` to the command for a bottleneck sweep. `power_norm` is taken from the config (verified in cell 6) — **nothing here overrides it**.

In [ ]:
import subprocess, sys, yaml

SEEDS = [0]          # single seed to save compute (configs default to [0, 1, 2])
# dc is left at each config's default (32). No sweep.

# Default: the two sqrt configs on both tasks (4 runs, 1 seed each, dc=32).
RUN_CONFIGS = [
    ('scl',     'cov_supervised_sqrt.yaml'),
    ('scl',     'attention_cov_sqrt.yaml'),
    ('meltome', 'cov_supervised_sqrt.yaml'),
    ('meltome', 'attention_cov_sqrt.yaml'),
    # Optional non-sqrt ablation baseline for attention_cov (uncomment to include):
    # ('scl',     'attention_cov.yaml'),
    # ('meltome', 'attention_cov.yaml'),
]

def stream(cmd, cwd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, cwd=str(cwd))
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed ({proc.returncode}): {" ".join(cmd)}')

for task, fname in RUN_CONFIGS:
    src = REPO_DIR / 'configs' / task / fname
    cfg = yaml.safe_load(src.read_text())
    cfg['seeds'] = SEEDS                                  # override to fewer seeds
    # Task-prefixed copy so scl & meltome don't collide on the shared stem.
    prefixed = src.with_name(f'{task}_{fname}')
    prefixed.write_text(yaml.safe_dump(cfg, sort_keys=False))
    cfg_rel = prefixed.relative_to(REPO_DIR)
    print(f'\n=== {cfg_rel}  (seeds={SEEDS}, dc={cfg["pooling"]["dc"]}) ===')
    cmd = [sys.executable, str(REPO_DIR / 'scripts' / 'run_experiment.py'),
           '--config', str(cfg_rel)]
    stream(cmd, cwd=REPO_DIR)

print('\nDone — all sqrt runs finished.')

## 8 · Mirror results to Drive

Copies every JSON in `results/` to Drive so it survives the session. These land in `RESULTS_OUT` next to the existing core-grid JSONs; re-running `scripts/make_plots.py` over that folder regenerates the figures + `summary_table.tsv` with the sqrt rows included.

In [ ]:
import shutil

src = REPO_DIR / 'results'
if not src.exists():
    print(f'No results dir at {src} — nothing to copy.')
else:
    RESULTS_OUT.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in src.rglob('*'):
        if p.is_file():
            rel = p.relative_to(src)
            dst = RESULTS_OUT / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            n += 1
    print(f'Copied {n} files to {RESULTS_OUT}')

## 9 · Verify `power_norm` ON + summary

Prints every sqrt result with its `power_norm` flag and headline metric, and **hard-fails** if any output had `power_norm` off — so you can trust these numbers are genuine C^{1/2} runs.

In [ ]:
import json

runs_dir = REPO_DIR / 'results' / 'runs'
sqrt_files = [p for p in sorted(runs_dir.glob('*sqrt*')) if not p.stem.endswith('_sweep')]

print(f'{"file":<56}{"method":<16}{"dc":<5}{"power_norm":<12}{"metric":<11}value')
print('-' * 110)
bad = []
for p in sqrt_files:
    d = json.loads(p.read_text())
    mkey = 'accuracy_mean' if d['task'] == 'classification' else 'spearman_r_mean'
    skey = mkey.replace('_mean', '_std')
    pn = d.get('power_norm', False)
    flag = 'ON' if pn else '!!! OFF !!!'
    if not pn:
        bad.append(p.name)
    print(f'{p.name:<56}{d["method"]:<16}{str(d.get("dc")):<5}{flag:<12}'
          f'{mkey.replace("_mean",""):<11}{d[mkey]:.4f} ± {d[skey]:.4f}')

assert not bad, f'power_norm was OFF in: {bad} — these are NOT valid sqrt runs.'
print(f'\n✓ {len(sqrt_files)} sqrt results, every one with power_norm = ON.')